In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
dataset_root = os.getenv("DATASET_ROOT")

print(f"{os.listdir(dataset_root)}")

/kaggle/input/datasets/shuangsong/numerai-data-v52
['features.json', 'train.parquet', 'validation.parquet']


In [45]:
import numpy as np
import random
import torch
import os

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [46]:
import pandas as pd
import gc
import torch

def load_optimized(path, features, flag, tv_ratio=0.8):
    df = pd.read_parquet(path, columns=['target', 'era'] + features, engine='pyarrow')
    df = df.iloc[::4]

    if df['era'].dtype == 'object':
        df['era'] = df['era'].str.replace('era', '').astype(int)
    else:
        df['era'] = df['era'].astype(int)

    df = df.astype('float32')
    unique_eras = np.sort(df['era'].unique())
    ckpt_era = unique_eras[int(len(unique_eras) * tv_ratio)]

    if flag == "train":
        mask = df['era'] <= ckpt_era
    elif flag == "val":
        mask = df['era'] > ckpt_era
        
    X = torch.from_numpy(df.loc[mask, features].values).float()
    y = torch.from_numpy(df.loc[mask, 'target'].values).float()
    
    del df
    gc.collect() 
    
    return X, y

In [47]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, size, dropout_rate=0.5):
        super(ResBlock, self).__init__()
        self.l = nn.Linear(size, size)
        self.bn = nn.BatchNorm1d(size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        identity = x
        out = self.l(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + identity

class NumeraiNet(nn.Module):
    def __init__(self, input_size):
        super(NumeraiNet, self).__init__()
        self.initial_layer = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )
        self.res_blocks = nn.Sequential(
            ResBlock(256, 0.5),
            ResBlock(256, 0.5)
        )
        self.output_layer = nn.Linear(256, 1)

    def forward(self, x):
        x = self.initial_layer(x)
        x = self.res_blocks(x)
        return torch.sigmoid(self.output_layer(x))

In [49]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loaded device:{device}")
TRAIN_PATH = os.path.join(dataset_root, "train.parquet")
VAL_PATH = os.path.join(dataset_root, "validation.parquet")
FEAT_PATH = os.path.join(dataset_root, "features.json")

with open(FEAT_PATH, 'r') as f:
    medium_features = json.load(f)['feature_sets']['medium']

X, y = load_optimized(TRAIN_PATH, medium_features, flag='train')
X_val, y_val = load_optimized(VAL_PATH, medium_features, flag='val')

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=4096, shuffle=True)
model = NumeraiNet(len(medium_features)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
criterion = nn.MSELoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max',      
    factor=0.5,     
    patience=3,          
    min_lr=1e-6      
)

epochs = 40
best_corr = -1.0
CURRENT_SAVE_PATH = None

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device).reshape(-1,1)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        preds_list = []
        val_loader = DataLoader(TensorDataset(X_val), batch_size=8192)
        for val_batch in val_loader:
            batch_data = val_batch[0].to(device)
            p = model(batch_data).cpu().detach().flatten().tolist()
            preds_list.extend(p)
    torch.cuda.empty_cache()
    actual_y = y_val.cpu().numpy().flatten()
    predicted_y = np.array(preds_list).flatten()

    corr = pd.Series(actual_y).corr(pd.Series(predicted_y), method='spearman')
    scheduler.step(corr)
    print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {total_loss/len(loader):.5f} | Val CORR: {corr:.4f}")

    if corr > best_corr:
        if CURRENT_SAVE_PATH is not None and os.path.exists(CURRENT_SAVE_PATH):
            os.remove(CURRENT_SAVE_PATH)
        best_corr = corr
        corr_suffix = f"{int(corr * 10000):04d}"
        CURRENT_SAVE_PATH = f"/kaggle/working/model_medium_{corr_suffix}.pth"
        torch.save(model.state_dict(), CURRENT_SAVE_PATH)
        print(f"New optimized model saved at: {CURRENT_SAVE_PATH} (CORR: {best_corr:.4f})")

Loaded device:cuda
Epoch  1/40 | Loss: 0.06376 | Val CORR: 0.0005
New optimized model saved at: /kaggle/working/model_medium_0004.pth (CORR: 0.0005)
Epoch  2/40 | Loss: 0.05609 | Val CORR: 0.0019
New optimized model saved at: /kaggle/working/model_medium_0019.pth (CORR: 0.0019)
Epoch  3/40 | Loss: 0.05396 | Val CORR: 0.0035
New optimized model saved at: /kaggle/working/model_medium_0034.pth (CORR: 0.0035)
Epoch  4/40 | Loss: 0.05263 | Val CORR: 0.0032
Epoch  5/40 | Loss: 0.05169 | Val CORR: 0.0040
New optimized model saved at: /kaggle/working/model_medium_0039.pth (CORR: 0.0040)
Epoch  6/40 | Loss: 0.05106 | Val CORR: 0.0049
New optimized model saved at: /kaggle/working/model_medium_0048.pth (CORR: 0.0049)
Epoch  7/40 | Loss: 0.05061 | Val CORR: 0.0056
New optimized model saved at: /kaggle/working/model_medium_0055.pth (CORR: 0.0056)
Epoch  8/40 | Loss: 0.05040 | Val CORR: 0.0052
Epoch  9/40 | Loss: 0.05013 | Val CORR: 0.0052
Epoch 10/40 | Loss: 0.05000 | Val CORR: 0.0059
New optimized